[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aditya-siraskar/PINN_FTRE/blob/main/notebooks/03_phase1_stage2c_ablations.ipynb)

# 03 · Stage 2c: ablations of the PINN design choices

**Phase:** Phase 1: validation (1-D)

**Goal:** Test each tail-specific design choice instead of assuming it helps (log residual, tail sampling, Gauss–Hermite nodes, normalisation).

**Outputs:** `configs/ablations/*.yaml`, `results/stage2_pinn1d_<tag>/`, `figures/stage2_ablation/fig01_ablation_summary`

> Runtime → Change runtime type → **T4 GPU**. Run the setup cell first. With `USE_DRIVE = True` the repo lives in
> `MyDrive/PINN_FTRE`, so figures and results persist. Core logic lives in `pinn_tailrisk/` and `scripts/`;
> this notebook only runs it and shows the results.

In [ ]:
# @title Setup: repo on Google Drive, install, GPU check  { display-mode: "form" }
USE_DRIVE = True   # keep figures/results across Colab sessions (recommended)
REPO_URL = "https://github.com/aditya-siraskar/PINN_FTRE"

import os, sys
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive"
else:
    BASE = "/content"
ROOT = f"{BASE}/PINN_FTRE"
if not os.path.exists(ROOT):
    !git clone -q {REPO_URL} "{ROOT}"
else:
    !git -C "{ROOT}" pull -q --ff-only
%cd "{ROOT}"
!pip install -q -e .
sys.path.insert(0, ROOT)

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (Runtime > Change runtime type > T4 GPU)")

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pinn_tailrisk import plotting as P
from pinn_tailrisk.notebook import show_figures, load_metrics
P.use_style()

## 1. Variants
Each variant changes **one** setting relative to `configs/stage2_pinn1d.yaml`. Lower `STEPS` for a quick pass.

In [ ]:
import yaml, copy, pathlib
STEPS = 20000
base = yaml.safe_load(open("configs/stage2_pinn1d.yaml"))
VARIANTS = {
    "baseline":       {},
    "linear_resid":   {"residual": "linear"},
    "no_tail_sampl":  {"tail_share": 0.0},
    "gh8":            {"n_gh": 8},
    "gh48":           {"n_gh": 48},
    "no_norm":        {"w_norm": 0.0},
}
pathlib.Path("configs/ablations").mkdir(exist_ok=True)
for tag, change in VARIANTS.items():
    c = copy.deepcopy(base); c["pinn"].update(change)
    yaml.safe_dump(c, open(f"configs/ablations/{tag}.yaml", "w"))
print(list(VARIANTS))

## 2. Train all variants
About 6 × the full-run time. Each finished variant is saved, so you can resume after a disconnect (finished ones are skipped).

In [ ]:
for tag in VARIANTS:
    if pathlib.Path(f"results/stage2_pinn1d_ab_{tag}/metrics.json").exists():
        print("skip", tag); continue
    !python scripts/stage2_pinn1d.py --config configs/ablations/{tag}.yaml --steps {STEPS} --tag ab_{tag}

## 3. Summary figure

In [ ]:
rows = []
for tag in VARIANTS:
    p = pathlib.Path(f"results/stage2_pinn1d_ab_{tag}/metrics.json")
    if not p.exists():
        continue
    m = load_metrics(f"stage2_pinn1d_ab_{tag}")
    r = pd.DataFrame(m["risk"])
    for h in (1, 10):
        q = r[(r.h == h) & (r.alpha == 0.99)].iloc[0]
        rows.append(dict(variant=tag, horizon=f"{h}-day", es99_abs_rel=abs(q.es_rel_pinn),
                         tail_log_mae=m["density_error"][f"{h}d"]["left_tail_mae_log"], train_min=m["train_time_s"] / 60))
ab = pd.DataFrame(rows)
display(ab.pivot(index="variant", columns="horizon", values=["es99_abs_rel", "tail_log_mae"]).style.format("{:.2e}"))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
order = [v for v in VARIANTS if v in set(ab.variant)]
y = np.arange(len(order))
for ax, col, xl in [(axes[0], "es99_abs_rel", "|ES₉₉ relative error|"), (axes[1], "tail_log_mae", "left-tail mean |log error|")]:
    for h, c, dy in [("1-day", P.SEQ_BLUE(0.45), -0.15), ("10-day", P.SEQ_BLUE(0.95), 0.15)]:
        s = ab[ab.horizon == h].set_index("variant").loc[order, col]
        ax.barh(y + dy, s.values, 0.28, color=c, label=h)
    ax.set(xscale="log", xlabel=xl); ax.grid(axis="y", visible=False)
axes[0].set(yticks=y, yticklabels=order); axes[0].invert_yaxis()
axes[0].legend(title="horizon"); P.panel_labels(axes); fig.tight_layout()
P.savefig(fig, "stage2_ablation", "fig01_ablation_summary")
show_figures("stage2_ablation")